In [1]:
from collections import Counter
from pathlib import Path

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.response_synthesizers import ResponseMode
from llama_index.core.vector_stores.types import (
    FilterOperator,
    MetadataFilter,
    MetadataFilters,
)
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding


def file_metadata(path: str) -> dict:
    """Tag each document with its ticker folder (data/AAPL/..., data/NVDA/...)."""
    p = Path(path).resolve()
    data_root = Path("data").resolve()
    try:
        rel = p.relative_to(data_root)
        folder = rel.parts[0] if rel.parts else ""
    except ValueError:
        folder = ""
    return {"file_path": str(p), "folder": folder}

# Cap num_ctx via context_window — reduces VRAM spikes vs full model ctx on big RAG prompts
Settings.llm = Ollama(
    model="llama3.1:8b",
    request_timeout=300.0,
    context_window=8192,
)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

documents = SimpleDirectoryReader(
    "./data", recursive=True, file_metadata=file_metadata
).load_data()
print("Loaded by folder:", Counter(d.metadata.get("folder", "?") for d in documents))

splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=128)
index = VectorStoreIndex.from_documents(documents, transformations=[splitter])

# No metadata filter: top-k is global — "risk" queries often match AAPL's huge Item 1A.
query_engine = index.as_query_engine(
    similarity_top_k=8,
    response_mode=ResponseMode.TREE_SUMMARIZE,
)

response = query_engine.query("What are the main points of the documents provided?")
print(response)

2026-05-16 14:45:35,830 - INFO - NumExpr defaulting to 16 threads.


Loaded by folder: Counter({'NVDA': 130, 'AAPL': 80})


2026-05-16 14:45:42,170 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-16 14:45:42,282 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-16 14:45:42,394 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-16 14:45:42,484 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-16 14:45:42,570 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-16 14:45:42,673 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-16 14:45:42,764 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-16 14:45:42,859 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-16 14:45:42,967 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-16 14:45:43,055 - INFO - HTTP Request: POST http://localhost:1143

The main points of the documents provided include:

* The first document appears to be a stock award agreement between NVIDIA Corporation (NVDA) and an individual, detailing the terms and conditions of a Restricted Stock Unit Award under the Amended & Restated 2007 Equity Incentive Plan.
* The second document is a filing with the U.S. Securities and Exchange Commission (SEC) by Apple Inc. (AAPL), including financial statements for the fiscal year ended September 27, 2025.
* Both documents are related to publicly traded companies and involve securities laws and regulations.

The key points of the stock award agreement include:

* The individual has been awarded a number of Restricted Stock Units that will vest according to a specified schedule
* The Award is subject to certain conditions, including the requirement for Continuous Service
* Vesting will cease upon termination of Continuous Service, except in cases of death or acceleration as described in the Grant Notice

The key points o

In [7]:
response = query_engine.query("What are the risk Apple facing?")
print(response)

2026-05-12 22:17:00,445 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-12 22:17:03,648 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Material adverse effects on the business, reputation, results of operations, financial condition, and stock price due to various risks including:

* Macroeconomic and industry risks such as economic conditions, consumer confidence, and spending
* Uncertainty about or decline in global or regional economic conditions
* Adverse reactions from stakeholders regarding social and other issues related to the business
* Legal proceedings and claims that arise in the ordinary course of business
* Interest rate risk due to fluctuations in US interest rates affecting investment portfolio and term debt
* Foreign exchange rate risk due to changes in exchange rates negatively affecting net sales, gross margins, and fair values of certain assets and liabilities

Additionally, Apple is exposed to risks such as:

* Volatility in stock price
* Business disruptions
* Climate change
* Concentration of sales to customers and partners
* Ability to attract, retain, and motivate executives and key employees
*

In [8]:
response = query_engine.query("What are the risk Nvidia facing?")
print(response)

2026-05-12 22:17:09,708 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-12 22:17:11,778 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Nvidia is facing risks related to its industry and markets, such as failure to meet evolving needs of customers, competition impacting market share and financial results. Additionally, there are risks related to demand, supply, and manufacturing, including long lead times, uncertain supplier relationships, product defects, and regulatory compliance.

Furthermore, Nvidia's business is exposed to risks associated with international sales and operations, including economic conditions and foreign exchange rate fluctuations. The company also faces risks related to cybersecurity, data breaches, and cyber-attacks, which could disrupt its operations and impact financial results.

Moreover, Nvidia has a concentration of sales to customers and partners, which makes it vulnerable to losing or being prevented from selling to any of these end-customers. The company may also face issues with attracting and retaining executives and key employees due to high expectations for future growth and profitab